# 부록 A. 참조 자료 종합 가이드

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_A_%EC%B0%B8%EC%A1%B0_%EC%9E%90%EB%A3%8C_%EC%A2%85%ED%95%A9_%EA%B0%80%EC%9D%B4%EB%93%9C.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## PyTorch 핵심 API 치트시트

**(1) 텐서 생성과 형태 변환.** 데이터를 담는 그릇을 만들고 모양을 바꾸는 기본기다.


In [ ]:
import torch
x = torch.tensor([[1., 2.], [3., 4.]])    # 리스트 를 텐서로
z = torch.zeros(3, 4); o = torch.ones(2)  # 0/1 로 채운 텐서
r = torch.randn(2, 3)                      # 표준정규 난수
a = torch.arange(0, 12).reshape(3, 4)      # 0~11 을 3x4 로
x.shape, x.dtype, x.device                 # 형태/자료형/장치 확인
x.view(-1)          # 1차원으로 펼치기(메모리 공유)
x.permute(1, 0)     # 축 순서 바꾸기(전치)
x.unsqueeze(0)      # 맨 앞에 크기 1 차원 추가
torch.cat([x, x], dim=0)     # 행 방향으로 이어 붙이기

**(2) 자동 미분(autograd).** 손실을 한 번 계산하면 기울기가 자동으로 흐른다.


In [ ]:
w = torch.tensor([2.0], requires_grad=True)  # 미분 대상
loss = (w * 3 - 6) ** 2       # 임의의 손실
loss.backward()               # 기울기 계산
print(w.grad)                 # dL/dw 확인
with torch.no_grad():         # 기울기 추적 끔(갱신/추론)
    w -= 0.1 * w.grad
w.grad.zero_()                # 다음 스텝 전 기울기 초기화

**(3) 모델 정의: `nn.Module**.` `forward`에 순전파만 적으면 역전파는 자동이다.


In [ ]:
import torch.nn as nn
class MLP(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.ReLU(),
            nn.Linear(d_hid, d_out))
    def forward(self, x):
        return self.net(x)        # 순전파만 정의
model = MLP(784, 128, 10)

**(4) 표준 학습 루프.** 3장부터 12장까지 거의 모든 실습의 뼈대다.


In [ ]:
# 학습 루프 시연용 합성 데이터 (입력 784차원, 10개 클래스)
from torch.utils.data import TensorDataset, DataLoader
EPOCHS = 1
model = model.to(device)
X_syn = torch.randn(256, 784)
y_syn = torch.randint(0, 10, (256,))
train_loader = DataLoader(TensorDataset(X_syn, y_syn), batch_size=64, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()              # (1) 기울기 초기화
        pred = model(xb)             # (2) 순전파
        loss = loss_fn(pred, yb)     # (3) 손실
        loss.backward()             # (4) 역전파
        opt.step()                   # (5) 가중치 갱신

**(5) 평가와 저장/불러오기.** 추론 시에는 `eval()`과 `no_grad()`를 함께 쓴다.


In [ ]:
model.eval()                         # 드롭아웃/BN 을 추론 모드로
with torch.no_grad():
    logits = model(xb.to(device))
    pred = logits.argmax(dim=1)      # 가장 큰 클래스 선택
torch.save(model.state_dict(), "model.pt")   # 파라미터만 저장(권장)
model.load_state_dict(torch.load("model.pt"))

## 실험 설계와 재현성

### 🔬 실습 — 두 모델의 통계적 비교

① **반복 점수 수집**: 같은 테스트 세트에 5개 시드로 각각 평가.


In [ ]:
import numpy as np
from scipy import stats
scores_A = [0.842, 0.851, 0.860, 0.848, 0.854]   # 베이스라인
scores_B = [0.868, 0.872, 0.879, 0.865, 0.876]   # 개선 모델

② **대응 t-검정**: 같은 시드 쌍에 대한 차이 검정.


In [ ]:
a, b = np.array(scores_A), np.array(scores_B)
t_stat, p_val = stats.ttest_rel(a, b)
print(f"A {a.mean():.4f}±{a.std():.4f} | B {b.mean():.4f}±{b.std():.4f}")
print(f"p-value {p_val:.4f}", "(유의미)" if p_val < 0.05 else "(유의하지 않음)")

③ **효과 크기(Cohen's d)**: 차이의 크기를 표준화.


In [ ]:
d = (b.mean() - a.mean()) / np.sqrt((a.std()**2 + b.std()**2)/2 + 1e-9)
print(f"Cohen's d {d:.3f}", "(대)" if abs(d)>0.8 else "(중)" if abs(d)>0.5 else "(소)")

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
